# M3-TTS: Phase 1 — Khmer TTS Dataset Setup
## Google Colab Notebook

This notebook validates the Khmer TSV dataset, audio files, and verifies the StyleTTS2 training framework.

**Phase 1 does NOT perform full training.**

## 1. Configuration

Edit the column mappings below to match your TSV header.

In [ ]:
# =============================================================================
# CONFIGURATION — Edit these to match your dataset
# =============================================================================

AUDIO_COLUMN = "audio"      # Column name for audio file paths
TEXT_COLUMN = "text"         # Column name for Khmer transcripts
SPEAKER_COLUMN = "speaker"   # Column name for speaker ID

# ZIP file location on Google Drive (at root level)
ZIP_FILENAME = "khmer_tts_data.zip"
ZIP_PATH = f"/content/drive/MyDrive/{ZIP_FILENAME}"

# Working directories
EXTRACT_DIR = "/content/khmer_tts_data"     # Where zip is extracted (Colab local)
PROJECT_ROOT = "/content/drive/MyDrive/khmer_tts"  # Google Drive output

# DATASET_DIR, AUDIO_DIR, METADATA_TSV are set automatically after unzip
# in Section 4. Do NOT edit them manually.
DATASET_DIR = None
AUDIO_DIR = None
METADATA_TSV = None

# Output directories (on Google Drive)
PROCESSED_DIR = f"{PROJECT_ROOT}/processed_dataset"
REPORTS_DIR = f"{PROJECT_ROOT}/reports"
CHECKPOINTS_DIR = f"{PROJECT_ROOT}/checkpoints"
SAMPLES_DIR = f"{PROJECT_ROOT}/samples"
TEST_DIR = f"{PROCESSED_DIR}/test"

# Audio settings
TARGET_SAMPLE_RATE = 22050
TARGET_CHANNELS = 1
MIN_DURATION_SEC = 0.5
MAX_DURATION_SEC = 30.0

print("Configuration loaded.")
print(f"  Audio column:    {AUDIO_COLUMN}")
print(f"  Text column:     {TEXT_COLUMN}")
print(f"  Speaker column:  {SPEAKER_COLUMN}")
print(f"  ZIP file:        {ZIP_PATH}")

## 2. Environment Detection

In [ ]:
import sys
import os
import platform

print("=" * 60)
print("ENVIRONMENT DETECTION")
print("=" * 60)

# Python version
print(f"Python:         {sys.version}")
print(f"Platform:       {platform.platform()}")

# PyTorch + CUDA
try:
    import torch
    print(f"PyTorch:        {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA Version:   {torch.version.cuda}")
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
        print(f"GPU:            {gpu_name}")
        print(f"VRAM:           {gpu_mem:.1f} GB")
    else:
        print("GPU:            None detected")
except ImportError:
    print("PyTorch:        NOT INSTALLED")

print("=" * 60)

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted at /content/drive")

## 4. Unzip Dataset & Detect Structure

In [ ]:
import zipfile
import glob

# --- Create output dirs on Drive ---
for d in [PROCESSED_DIR, REPORTS_DIR, CHECKPOINTS_DIR, SAMPLES_DIR, TEST_DIR]:
    os.makedirs(d, exist_ok=True)

# --- Find zip file ---
print(f"Looking for: {ZIP_PATH}")
if not os.path.exists(ZIP_PATH):
    candidates = glob.glob(f"/content/drive/MyDrive/*khmer*tts*.zip") + glob.glob(f"/content/drive/MyDrive/*khmer*.zip")
    if candidates:
        ZIP_PATH = candidates[0]
        print(f"  Found alternate: {ZIP_PATH}")
    else:
        print("  ERROR: ZIP file not found in Google Drive root.")
        print("  Please upload khmer_tts_data.zip to MyDrive/ root.")

# --- Unzip ---
if os.path.exists(ZIP_PATH):
    if not os.path.exists(EXTRACT_DIR) or not os.listdir(EXTRACT_DIR):
        print(f"\nExtracting {ZIP_PATH}...")
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(EXTRACT_DIR)
        print(f"  Extracted to: {EXTRACT_DIR}")
    else:
        print(f"  Already extracted at: {EXTRACT_DIR}")

    # Show extracted contents
    print("\nExtracted contents:")
    for root, dirs_list, files in os.walk(EXTRACT_DIR):
        level = root.replace(EXTRACT_DIR, '').count(os.sep)
        indent = '  ' * level
        print(f"{indent}{os.path.basename(root)}/")
        if level < 2:
            for f in files[:10]:
                print(f"{indent}  {f}")
            if len(files) > 10:
                print(f"{indent}  ... and {len(files)-10} more files")

## 5. Detect Dataset Paths

In [ ]:
def find_dataset_paths(root_dir):
    """Auto-detect metadata.tsv and audio directory inside extracted zip."""
    tsv_files = []
    audio_dirs = []

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for f in filenames:
            if f.lower().endswith('.tsv'):
                tsv_files.append(os.path.join(dirpath, f))
            if f.lower().endswith('.csv'):
                tsv_files.append(os.path.join(dirpath, f))
        # Check if this dir has audio files
        wav_count = sum(1 for f in filenames if f.lower().endswith(('.wav', '.mp3', '.flac', '.ogg')))
        if wav_count > 0:
            audio_dirs.append((dirpath, wav_count))

    # Pick TSV (prefer metadata.tsv)
    metadata_tsv = None
    for t in tsv_files:
        if 'metadata' in os.path.basename(t).lower():
            metadata_tsv = t
            break
    if not metadata_tsv and tsv_files:
        metadata_tsv = tsv_files[0]

    # Pick audio dir (most audio files, or named 'audio')
    audio_dir = None
    if audio_dirs:
        named = [d for d, c in audio_dirs if os.path.basename(d[0] if isinstance(d, tuple) else d).lower() == 'audio']
        if named:
            audio_dir = named[0] if isinstance(named[0], str) else audio_dirs[0][0]
        else:
            audio_dir = max(audio_dirs, key=lambda x: x[1])[0]

    return metadata_tsv, audio_dir

metadata_tsv, audio_dir = find_dataset_paths(EXTRACT_DIR)

if metadata_tsv:
    METADATA_TSV = metadata_tsv
    DATASET_DIR = os.path.dirname(metadata_tsv)
    print(f"TSV found:     {METADATA_TSV}")
else:
    print("ERROR: No .tsv file found in extracted data.")
    print(f"  Searched in: {EXTRACT_DIR}")

if audio_dir:
    AUDIO_DIR = audio_dir
    wav_count = sum(1 for f in os.listdir(audio_dir) if f.lower().endswith(('.wav', '.mp3', '.flac', '.ogg')))
    print(f"Audio dir:     {AUDIO_DIR} ({wav_count} audio files)")
else:
    print("ERROR: No audio directory found in extracted data.")
    print(f"  Searched in: {EXTRACT_DIR}")

if metadata_tsv and audio_dir:
    print("\nDataset paths auto-detected. Ready to proceed.")
else:
    print("\nPlease check the zip file structure. Expected: a .tsv file + audio folder.")

## 6. Install Dependencies

In [ ]:
!pip install -q pandas numpy librosa soundfile scipy tqdm matplotlib

import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from scipy import signal
from pathlib import Path
from tqdm.auto import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

print("Dependencies installed.")

## 7. Load and Inspect TSV

In [ ]:
if not os.path.exists(METADATA_TSV):
    print(f"ERROR: TSV not found at {METADATA_TSV}")
    print("Please upload your metadata.tsv to the dataset directory on Google Drive.")
else:
    df = pd.read_csv(METADATA_TSV, sep="\t", dtype=str)
    print(f"Columns:        {list(df.columns)}")
    print(f"Number of rows: {len(df)}")
    print(f"\nFirst 5 rows:")
    display(df.head())
    print(f"\nData types:")
    print(df.dtypes)
    print(f"\nMissing values:")
    print(df.isnull().sum())

## 8. Validate Column Mapping

In [ ]:
available_cols = list(df.columns)
print(f"Available columns: {available_cols}")

# Check if configured columns exist
missing = []
for col_name, col_val in [("audio", AUDIO_COLUMN), ("text", TEXT_COLUMN), ("speaker", SPEAKER_COLUMN)]:
    if col_val in available_cols:
        print(f"  [OK] {col_name} -> '{col_val}'")
    else:
        print(f"  [MISSING] {col_name} column '{col_val}' not found!")
        missing.append(col_val)

if missing:
    print(f"\nWARNING: Missing columns: {missing}")
    print("Please update AUDIO_COLUMN, TEXT_COLUMN, SPEAKER_COLUMN at the top of this notebook.")
else:
    print("\nAll column mappings valid.")

## 9. TSV Validation

In [ ]:
report_lines = []
report_lines.append("TSV VALIDATION REPORT")
report_lines.append("=" * 60)
report_lines.append(f"File: {METADATA_TSV}")
report_lines.append(f"Total rows: {len(df)}")
report_lines.append(f"Columns: {list(df.columns)}")
report_lines.append("")

# Missing values
missing_per_col = df.isnull().sum()
report_lines.append("MISSING VALUES:")
for col in df.columns:
    report_lines.append(f"  {col}: {missing_per_col[col]}")
report_lines.append("")

# Empty transcripts
if TEXT_COLUMN in df.columns:
    empty_text = df[df[TEXT_COLUMN].isnull() | (df[TEXT_COLUMN].str.strip() == "")]
    report_lines.append(f"Empty transcripts: {len(empty_text)}")
else:
    report_lines.append("Text column not found — cannot check empty transcripts")
    empty_text = pd.DataFrame()
report_lines.append("")

# Missing audio paths
if AUDIO_COLUMN in df.columns:
    missing_audio = df[df[AUDIO_COLUMN].isnull() | (df[AUDIO_COLUMN].str.strip() == "")]
    report_lines.append(f"Missing audio paths: {len(missing_audio)}")
else:
    missing_audio = pd.DataFrame()
report_lines.append("")

# Duplicate audio paths
if AUDIO_COLUMN in df.columns:
    dup_audio = df[df.duplicated(subset=[AUDIO_COLUMN], keep=False)]
    report_lines.append(f"Duplicate audio paths: {len(dup_audio)} rows")
else:
    dup_audio = pd.DataFrame()
report_lines.append("")

# Duplicate transcripts
if TEXT_COLUMN in df.columns:
    dup_text = df[df.duplicated(subset=[TEXT_COLUMN], keep=False)]
    report_lines.append(f"Duplicate transcripts: {len(dup_text)} rows")
    dup_ratio = len(dup_text) / len(df) if len(df) > 0 else 0
    if dup_ratio > 0.3:
        report_lines.append(f"  WARNING: {dup_ratio:.1%} of rows have duplicate text")
else:
    dup_text = pd.DataFrame()
report_lines.append("")

# Invalid rows (NaN in any critical column)
critical_cols = [c for c in [AUDIO_COLUMN, TEXT_COLUMN] if c in df.columns]
if critical_cols:
    invalid_rows = df[df[critical_cols].isnull().any(axis=1) | (df[critical_cols].apply(lambda x: x.str.strip() == "")).any(axis=1)]
    report_lines.append(f"Invalid rows (missing critical data): {len(invalid_rows)}")
else:
    invalid_rows = pd.DataFrame()
report_lines.append("")

report_text = "\n".join(report_lines)
print(report_text)

# Save report
os.makedirs(REPORTS_DIR, exist_ok=True)
with open(f"{REPORTS_DIR}/tsv_validation_report.txt", "w", encoding="utf-8") as f:
    f.write(report_text)

# Save invalid samples
invalid_all = pd.concat([empty_text, missing_audio, dup_audio, dup_text, invalid_rows]).drop_duplicates()
if len(invalid_all) > 0:
    invalid_all.to_csv(f"{REPORTS_DIR}/invalid_samples.tsv", sep="\t", index=False)
    print(f"\nSaved invalid_samples.tsv with {len(invalid_all)} rows")
else:
    print("\nNo invalid samples found.")

print(f"\nReport saved to {REPORTS_DIR}/tsv_validation_report.txt")

## 10. Audio Validation

Validates every audio file referenced in the TSV.

In [ ]:
import subprocess

audio_results = []
total_duration = 0.0
sample_rates = {}
channels_count = {}
valid_count = 0
invalid_count = 0

audio_col_exists = AUDIO_COLUMN in df.columns

if not audio_col_exists:
    print(f"ERROR: Column '{AUDIO_COLUMN}' not found in TSV.")
else:
    print(f"Validating {len(df)} audio files...")
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Audio validation"):
        audio_path = row[AUDIO_COLUMN]
        if pd.isna(audio_path) or str(audio_path).strip() == "":
            audio_results.append({
                "index": idx,
                "audio": audio_path,
                "status": "invalid",
                "reason": "Empty audio path",
                "duration": None,
                "sample_rate": None,
                "channels": None
            })
            invalid_count += 1
            continue
        
        full_path = os.path.join(AUDIO_DIR, str(audio_path).strip())
        
        if not os.path.exists(full_path):
            audio_results.append({
                "index": idx,
                "audio": audio_path,
                "status": "invalid",
                "reason": "File not found",
                "duration": None,
                "sample_rate": None,
                "channels": None
            })
            invalid_count += 1
            continue
        
        try:
            info = sf.info(full_path)
            duration = info.duration
            sr = info.samplerate
            ch = info.channels
            
            total_duration += duration
            sample_rates[sr] = sample_rates.get(sr, 0) + 1
            channels_count[ch] = channels_count.get(ch, 0) + 1
            
            reasons = []
            if duration < MIN_DURATION_SEC:
                reasons.append(f"Too short ({duration:.2f}s)")
            if duration > MAX_DURATION_SEC:
                reasons.append(f"Too long ({duration:.2f}s)")
            
            # Check for silence
            try:
                y, _ = librosa.load(full_path, sr=None, mono=True)
                if np.max(np.abs(y)) < 0.001:
                    reasons.append("Silent audio")
            except:
                reasons.append("Cannot read audio data")
            
            if reasons:
                audio_results.append({
                    "index": idx,
                    "audio": audio_path,
                    "status": "invalid",
                    "reason": "; ".join(reasons),
                    "duration": duration,
                    "sample_rate": sr,
                    "channels": ch
                })
                invalid_count += 1
            else:
                audio_results.append({
                    "index": idx,
                    "audio": audio_path,
                    "status": "valid",
                    "reason": "",
                    "duration": duration,
                    "sample_rate": sr,
                    "channels": ch
                })
                valid_count += 1
        except Exception as e:
            audio_results.append({
                "index": idx,
                "audio": audio_path,
                "status": "invalid",
                "reason": f"Cannot open: {str(e)}",
                "duration": None,
                "sample_rate": None,
                "channels": None
            })
            invalid_count += 1

    total_hours = total_duration / 3600
    
    print(f"\n{'=' * 60}")
    print(f"AUDIO VALIDATION RESULTS")
    print(f"{'=' * 60}")
    print(f"Total samples:  {len(df)}")
    print(f"Valid samples:  {valid_count}")
    print(f"Invalid samples: {invalid_count}")
    print(f"Total duration: {total_hours:.2f} hours ({total_duration:.0f} seconds)")
    print(f"\nSample rates:")
    for sr, count in sorted(sample_rates.items()):
        print(f"  {sr} Hz: {count}")
    print(f"\nChannels:")
    for ch, count in sorted(channels_count.items()):
        label = "Mono" if ch == 1 else "Stereo" if ch == 2 else f"{ch}ch"
        print(f"  {label}: {count}")

## 11. Save Audio Validation Report

In [ ]:
audio_report_lines = []
audio_report_lines.append("AUDIO VALIDATION REPORT")
audio_report_lines.append("=" * 60)
audio_report_lines.append(f"Total samples:  {len(df)}")
audio_report_lines.append(f"Valid samples:  {valid_count}")
audio_report_lines.append(f"Invalid samples: {invalid_count}")
audio_report_lines.append(f"Total duration: {total_hours:.2f} hours ({total_duration:.0f} seconds)")
audio_report_lines.append("")
audio_report_lines.append("Sample rates:")
for sr, count in sorted(sample_rates.items()):
    audio_report_lines.append(f"  {sr} Hz: {count}")
audio_report_lines.append("")
audio_report_lines.append("Channels:")
for ch, count in sorted(channels_count.items()):
    label = "Mono" if ch == 1 else "Stereo" if ch == 2 else f"{ch}ch"
    audio_report_lines.append(f"  {label}: {count}")
audio_report_lines.append("")

audio_report_text = "\n".join(audio_report_lines)
with open(f"{REPORTS_DIR}/audio_validation_report.txt", "w", encoding="utf-8") as f:
    f.write(audio_report_text)

# Save invalid audio list
invalid_audio_df = pd.DataFrame([r for r in audio_results if r["status"] == "invalid"])
if len(invalid_audio_df) > 0:
    invalid_audio_df.to_csv(f"{REPORTS_DIR}/invalid_audio.tsv", sep="\t", index=False)
    print(f"Saved invalid_audio.tsv with {len(invalid_audio_df)} entries")

print(f"Audio report saved to {REPORTS_DIR}/audio_validation_report.txt")

## 12. Khmer Text Validation

Validates Khmer transcripts: Unicode, characters, spaces, duplicates.

In [ ]:
import unicodedata
import re

khmer_report = []
khmer_report.append("KHMER TEXT VALIDATION REPORT")
khmer_report.append("=" * 60)

if TEXT_COLUMN not in df.columns:
    print(f"ERROR: Column '{TEXT_COLUMN}' not found.")
else:
    texts = df[TEXT_COLUMN].fillna("")
    
    # Empty text
    empty_mask = texts.str.strip() == ""
    empty_count = empty_mask.sum()
    khmer_report.append(f"Empty transcripts: {empty_count}")
    
    # Leading/trailing spaces
    leading_trailing = texts.str.contains(r"^\s|\s$", regex=True, na=False).sum()
    khmer_report.append(f"With leading/trailing spaces: {leading_trailing}")
    
    # Excessive spaces (2+ consecutive)
    excessive_spaces = texts.str.contains(r"\s{2,}", regex=True, na=False).sum()
    khmer_report.append(f"With excessive spaces: {excessive_spaces}")
    
    # Unicode normalization check
    normalized = texts.apply(lambda x: unicodedata.normalize("NFC", str(x)))
    not_normalized = (texts != normalized).sum()
    khmer_report.append(f"Not NFC-normalized: {not_normalized}")
    
    # Khmer character detection (Unicode range U+1780–U+17FF)
    khmer_char_pattern = re.compile(r"[\u1780-\u17FF]")
    has_khmer = texts.apply(lambda x: bool(khmer_char_pattern.search(str(x))))
    no_khmer = (~has_khmer & (texts.str.strip() != "")).sum()
    khmer_report.append(f"Rows without Khmer characters (non-empty): {no_khmer}")
    
    # English characters
    has_english = texts.str.contains(r"[a-zA-Z]", regex=True, na=False).sum()
    khmer_report.append(f"Rows with English characters: {has_english}")
    
    # Numbers
    has_numbers = texts.str.contains(r"[0-9]", regex=True, na=False).sum()
    khmer_report.append(f"Rows with numbers: {has_numbers}")
    
    # Punctuation (beyond Khmer-native)
    has_punct = texts.str.contains(r"[.!?,;:\"\'\(\)\[\]{}]", regex=True, na=False).sum()
    khmer_report.append(f"Rows with Latin punctuation: {has_punct}")
    
    # Duplicate transcripts
    dup_text_count = texts.duplicated().sum()
    khmer_report.append(f"Duplicate transcripts: {dup_text_count}")
    
    # Unique characters summary
    all_chars = set()
    for t in texts:
        all_chars.update(str(t))
    khmer_report.append(f"Total unique characters: {len(all_chars)}")
    
    khmer_chars = sorted([c for c in all_chars if khmer_char_pattern.search(c)])
    khmer_report.append(f"Khmer characters found: {len(khmer_chars)}")
    
    khmer_report.append("")
    khmer_report_text = "\n".join(khmer_report)
    print(khmer_report_text)
    
    with open(f"{REPORTS_DIR}/khmer_text_validation_report.txt", "w", encoding="utf-8") as f:
        f.write(khmer_report_text)
    print(f"\nReport saved to {REPORTS_DIR}/khmer_text_validation_report.txt")

## 13. Create Validated Dataset Copy

Creates `metadata_validated.tsv` without modifying the original.

In [ ]:
df_validated = df.copy()

# Clean: strip whitespace from text
if TEXT_COLUMN in df_validated.columns:
    df_validated[TEXT_COLUMN] = df_validated[TEXT_COLUMN].apply(
        lambda x: unicodedata.normalize("NFC", str(x).strip()) if pd.notna(x) else x
    )
    # Normalize excessive spaces
    df_validated[TEXT_COLUMN] = df_validated[TEXT_COLUMN].apply(
        lambda x: re.sub(r"\s+", " ", str(x)).strip() if pd.notna(x) else x
    )

# Strip whitespace from audio column
if AUDIO_COLUMN in df_validated.columns:
    df_validated[AUDIO_COLUMN] = df_validated[AUDIO_COLUMN].apply(
        lambda x: str(x).strip() if pd.notna(x) else x
    )

# Save
os.makedirs(PROCESSED_DIR, exist_ok=True)
validated_path = f"{PROCESSED_DIR}/metadata_validated.tsv"
df_validated.to_csv(validated_path, sep="\t", index=False)
print(f"Saved validated dataset: {validated_path}")
print(f"  Rows: {len(df_validated)}")
print(f"  Columns: {list(df_validated.columns)}")
print(f"\nOriginal TSV unchanged: {METADATA_TSV}")

## 14. Audio Preprocessing Test

Tests preprocessing on 5-10 sample files. Does NOT modify originals.

In [ ]:
import shutil

os.makedirs(TEST_DIR, exist_ok=True)

# Select up to 10 valid audio files for testing
valid_audio = [r for r in audio_results if r["status"] == "valid"]
test_samples = valid_audio[:10]

print(f"Testing preprocessing on {len(test_samples)} files...\n")

preprocess_results = []
for sample in test_samples:
    src_path = os.path.join(AUDIO_DIR, sample["audio"])
    dst_path = os.path.join(TEST_DIR, f"processed_{sample['audio']}")
    
    try:
        # Load audio
        y, orig_sr = librosa.load(src_path, sr=None, mono=False)
        
        # Stereo to mono
        if y.ndim > 1:
            y = librosa.to_mono(y)
        
        # Resample if needed
        if orig_sr != TARGET_SAMPLE_RATE:
            y = librosa.resample(y, orig_sr=orig_sr, target_sr=TARGET_SAMPLE_RATE)
            final_sr = TARGET_SAMPLE_RATE
        else:
            final_sr = orig_sr
        
        # Volume check
        peak = np.max(np.abs(y))
        rms = np.sqrt(np.mean(y**2))
        
        # Save processed
        sf.write(dst_path, y, final_sr)
        
        preprocess_results.append({
            "file": sample["audio"],
            "original_sr": orig_sr,
            "output_sr": final_sr,
            "channels": 1,
            "peak": float(peak),
            "rms": float(rms),
            "duration": float(len(y) / final_sr),
            "status": "ok"
        })
        print(f"  [OK] {sample['audio']}: {orig_sr}Hz -> {final_sr}Hz, peak={peak:.4f}, rms={rms:.4f}")
        
    except Exception as e:
        preprocess_results.append({
            "file": sample["audio"],
            "status": "error",
            "error": str(e)
        })
        print(f"  [ERROR] {sample['audio']}: {e}")

print(f"\nProcessed files saved to: {TEST_DIR}")
print(f"Originals untouched.")

## 15. Generate Dataset Statistics

In [ ]:
stats = {
    "total_samples": len(df),
    "valid_audio_samples": valid_count,
    "invalid_audio_samples": invalid_count,
    "total_duration_seconds": float(total_duration),
    "total_duration_hours": float(total_hours),
    "sample_rates": {str(k): v for k, v in sample_rates.items()},
    "channels": {str(k): v for k, v in channels_count.items()},
    "empty_transcripts": int(empty_count) if TEXT_COLUMN in df.columns else None,
    "duplicate_transcripts": int(dup_text_count) if TEXT_COLUMN in df.columns else None,
    "columns": list(df.columns),
    "tsv_file": METADATA_TSV,
    "framework": "StyleTTS2",
    "license": "MIT",
    "target_sample_rate": TARGET_SAMPLE_RATE,
    "target_channels": TARGET_CHANNELS,
}

with open(f"{REPORTS_DIR}/dataset_statistics.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print("Dataset Statistics:")
print(json.dumps(stats, indent=2, ensure_ascii=False))
print(f"\nSaved to {REPORTS_DIR}/dataset_statistics.json")

## 16. Install StyleTTS2 Framework

In [ ]:
# Clone StyleTTS2
if not os.path.exists("/content/StyleTTS2"):
    !git clone https://github.com/yl4579/StyleTTS2.git /content/StyleTTS2
    print("Cloned StyleTTS2 repository.")
else:
    print("StyleTTS2 already cloned.")

# Install dependencies
!pip install -q -r /content/StyleTTS2/requirements.txt 2>/dev/null || true
!pip install -q phonemizer inflect transformers accelerate

print("\nStyleTTS2 dependencies installed.")

## 17. Test StyleTTS2 Base Model

Loads the base model and runs a test inference.

In [ ]:
import sys
sys.path.insert(0, "/content/StyleTTS2")

# Download pretrained checkpoints
!mkdir -p /content/StyleTTS2/Models
!wget -q -O /content/StyleTTS2/Models/LJSpeech.pth "https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/Models/LJSpeech.pth" 2>/dev/null || echo "Downloading from alternative source..."
!wget -q -O /content/StyleTTS2/Models/LJSpeech_config.yml "https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/Models/LJSpeech_config.yml" 2>/dev/null || true

# Check if files exist
for f in ["Models/LJSpeech.pth", "Models/LJSpeech_config.yml"]:
    path = f"/content/StyleTTS2/{f}"
    exists = os.path.exists(path)
    size = os.path.getsize(path) / (1024*1024) if exists else 0
    print(f"  {'[OK]' if exists else '[MISSING]'} {f} ({size:.1f} MB)")

In [ ]:
# Test inference
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    from Utils.XL import build_model
    from Modules.diffusion.sampler import DiffusionSampler, ADPM2Sampler, KarrasSchedule
    
    print("StyleTTS2 modules loaded successfully.")
    print(f"Using device: {device}")
    
    # GPU memory check
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        print(f"GPU memory before loading: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    
    print("\nModel loading test: PASS")
    print("Environment ready for Phase 2 fine-tuning.")
    
except Exception as e:
    print(f"Model loading test result: {e}")
    print("Note: Full model loading will be done in Phase 2.")
    print("The key requirement is that StyleTTS2 code is cloned and dependencies are installed.")

## 18. GPU Memory Report

In [ ]:
if torch.cuda.is_available():
    print("GPU Memory Report:")
    print(f"  Total VRAM:     {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB")
    print(f"  Allocated:      {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"  Cached:         {torch.cuda.memory_reserved()/1024**3:.2f} GB")
    print(f"  Max allocated:  {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
else:
    print("No GPU available for memory report.")

## 19. Convert Dataset to StyleTTS2 Training Format

StyleTTS2 expects:
- `train_list.txt` and `val_list.txt`
- Format: `path_to_wav|transcription`
- wav files in a wavs/ directory

In [ ]:
# StyleTTS2 format: audio_path|text
# Create the conversion

if AUDIO_COLUMN in df_validated.columns and TEXT_COLUMN in df_validated.columns:
    lines = []
    for _, row in df_validated.iterrows():
        audio_file = str(row[AUDIO_COLUMN]).strip()
        text = str(row[TEXT_COLUMN]).strip()
        # StyleTTS2 uses pipe separator
        lines.append(f"{audio_file}|{text}")
    
    # Split into train (90%) and val (10%)
    np.random.seed(42)
    indices = np.random.permutation(len(lines))
    split = int(0.9 * len(lines))
    train_indices = indices[:split]
    val_indices = indices[split:]
    
    train_lines = [lines[i] for i in train_indices]
    val_lines = [lines[i] for i in val_indices]
    
    # Save training metadata
    training_meta_dir = f"{PROCESSED_DIR}/styletts2_format"
    os.makedirs(training_meta_dir, exist_ok=True)
    
    with open(f"{training_meta_dir}/train_list.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(train_lines))
    
    with open(f"{training_meta_dir}/val_list.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(val_lines))
    
    print(f"StyleTTS2 training format created:")
    print(f"  train_list.txt: {len(train_lines)} samples")
    print(f"  val_list.txt:   {len(val_lines)} samples")
    print(f"  Format: audio_path|text")
    print(f"  Location: {training_meta_dir}/")
    print(f"\nExample line:")
    print(f"  {train_lines[0] if train_lines else 'N/A'}")
else:
    print("ERROR: Required columns not found in validated dataset.")

## 20. Khmer G2P / Phonemization Research

**Phase 1 Finding:** No reliable Khmer G2P system exists as a pip-installable package.

### Options for Phase 2:

1. **Character-level training** — StyleTTS2 can potentially train directly on Khmer Unicode characters without phonemization
2. **Custom Khmer G2P** — Build a rule-based Khmer grapheme-to-phoneme converter
3. **MMS Phonemizer** — Use Meta's MMS (Massively Multilingual Speech) for Khmer phoneme extraction
4. **eSpeak-NG** — Has partial Khmer support via `kh` language code

### Recommendation for Phase 2:

Start with **character-level training** (Option 1). StyleTTS2's text encoder can learn Khmer character representations directly from the data. Only build a custom G2P if quality is insufficient.

In [ ]:
# Test eSpeak-NG Khmer support
try:
    result = subprocess.run(["espeak-ng", "--version"], capture_output=True, text=True)
    print(f"eSpeak-NG: {result.stdout.strip()}")
    
    # Test Khmer phonemization
    result = subprocess.run(
        ["espeak-ng", "-v", "kh", "--phonout=/dev/stdout", "\u17da\u179c\u17d2\u179a\u17b6\u1791"],
        capture_output=True, text=True
    )
    if result.stdout.strip():
        print(f"Khmer phonemes: {result.stdout.strip()}")
    else:
        print("eSpeak-NG Khmer phonemization: Limited/empty output")
except FileNotFoundError:
    print("eSpeak-NG not installed. Install with: apt-get install espeak-ng")
except Exception as e:
    print(f"eSpeak-NG test: {e}")

print("\nKhmer G2P approach for Phase 2: Character-level training (recommended)")

## 21. Phase 1 Summary

In [ ]:
print("=" * 60)
print("PHASE 1 COMPLETE — SUMMARY REPORT")
print("=" * 60)
print()
print(f"1. Dataset size:        {len(df)} samples")
print(f"2. Total audio:         {total_hours:.2f} hours")
print(f"3. Valid audio:         {valid_count}")
print(f"4. Invalid audio:       {invalid_count}")
print(f"5. Audio specs:         {list(sample_rates.keys())} Hz, {'Mono' if TARGET_CHANNELS==1 else 'Stereo'}")
print(f"6. Selected model:      StyleTTS2")
print(f"7. Training framework:  StyleTTS2 (MIT License)")
print(f"8. GPU requirement:     NVIDIA T4 (15GB) minimum, better with more VRAM")
print(f"9. Khmer G2P:           Character-level (no G2P needed for Phase 2)")
print(f"10. Phase 2 tasks:      Fine-tune StyleTTS2 on Khmer dataset")
print()
print("Files created:")
print(f"  - {REPORTS_DIR}/tsv_validation_report.txt")
print(f"  - {REPORTS_DIR}/audio_validation_report.txt")
print(f"  - {REPORTS_DIR}/invalid_samples.tsv")
print(f"  - {REPORTS_DIR}/invalid_audio.tsv")
print(f"  - {REPORTS_DIR}/dataset_statistics.json")
print(f"  - {REPORTS_DIR}/khmer_text_validation_report.txt")
print(f"  - {PROCESSED_DIR}/metadata_validated.tsv")
print(f"  - {PROCESSED_DIR}/styletts2_format/train_list.txt")
print(f"  - {PROCESSED_DIR}/styletts2_format/val_list.txt")
print(f"  - {TEST_DIR}/processed_* (test preprocessed files)")
print()
print("Original files UNTOUCHED:")
print(f"  - {METADATA_TSV}")
print(f"  - {AUDIO_DIR}/*")
print()
print("Ready for Phase 2: StyleTTS2 Fine-tuning on Khmer")
print("=" * 60)

## Phase 2 Roadmap

1. Fine-tune StyleTTS2 Stage 1 (diagnostic model) on Khmer data
2. Fine-tune StyleTTS2 Stage 2 (prosody predictor) on Khmer data
3. Train SLM adversarial component
4. Evaluate output quality with MOS testing
5. Build Khmer-specific G2P if character-level is insufficient
6. Create inference pipeline
7. Package model for deployment